# Coordinated simulation progress and results

This notebook reports the current on-disk status of `star_3` and `linear_A_to_D` under `data/simulation/coord_diff`, and computes the requested paper-text values. Re-run the cells to refresh the snapshot.

**Progress definitions.** A generated tree has both `nodes.csv` and `edges.csv`. Configured targets from the [Snakefile](../Snakefile) are 100 generated settings, 10 batches per setting, 128 trees per star batch, and 10 trees per linear batch. CONCERT and diversitree are batch-level fits; the current linear fits use the first 8 trees, star CONCERT uses the first 8 trees, and star diversitree uses all generated trees in its batch. BEAM and MACH2 are tree-level runs. An inference is counted only if its output parses and passes method-specific completion checks.

**Transition-order metrics.** For each batch, compare the three directed transitions (A→B, A→C, A→D for star; A→B, B→C, C→D for linear), pairing true probabilities with inferred CONCERT probabilities or diversitree/BEAM rates. BEAM rates are averaged across completed tree-level runs within each batch. Spearman correlation is undefined when inferred values are constant. The wrong-order rate is the fraction of comparable transition pairs ranked in the opposite order; inferred ties count as half an error, and true-probability ties are omitted. Each batch has at most three pairwise comparisons, so these metrics are coarse.

In [1]:
from collections import defaultdict
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "Snakefile").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repository root from the notebook directory")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
DATA_ROOT = REPO_ROOT / "data" / "simulation" / "coord_diff"

from src.plot_star_diversitree_errors import (
    LINEAR_BEAM_RATE_COLUMNS,
    _read_beam_rates as read_linear_beam_rates,
    _read_mach2_tree_transition_time_error,
    load_matched_absolute_errors,
    load_matched_linear_absolute_errors,
    load_mach2_transition_time_errors,
    load_star_transition_time_errors,
)
from src.plot_linear_transition_inference_accuracy import (
    LINEAR_TRANSITIONS,
    _read_diversitree_estimates as read_linear_diversitree,
    _read_ground_truth as read_linear_truth,
    _read_joint_estimates as read_linear_concert,
    load_concert_transition_time_errors,
)
from src.plot_star_inference_accuracy import load_completed_progressive_comparisons
from src.plot_star_transition_rates_vs_probabilities import (
    _read_beam_rates as read_star_beam_rates,
    _read_diversitree_rates as read_star_diversitree,
    _read_ground_truth as read_star_truth,
)

TOPOLOGY_SPECS = {
    "star_3": {"topology": "star", "trees_per_batch": 128},
    "linear_A_to_D": {"topology": "linear", "trees_per_batch": 10},
}
TRANSITIONS = ("B", "C", "D")
BATCHES_PER_SETTING = 10
FIRST_EIGHT = 8


def numeric_suffix(path, prefix):
    suffix = path.name.removeprefix(prefix)
    return int(suffix) if suffix.isdigit() else -1


def add_fit(fits, correlations, topology, method, setting, batch, trees_used,
            *, tree=None, no_transitions=False, true_values=None, estimates=None):
    fits.append({
        "topology": topology, "method": method, "setting": setting,
        "batch": batch, "tree": tree, "trees_used": int(trees_used),
        "no_predicted_transitions": bool(no_transitions),
    })


def scan_simulation(dataset):
    spec = TOPOLOGY_SPECS[dataset]
    topology = spec["topology"]
    data_dir = DATA_ROOT / dataset
    settings_csv = pd.read_csv(data_dir / "generated_settings.csv")
    truth_by_setting = (
        read_star_truth(data_dir) if topology == "star"
        else read_linear_truth(data_dir / "generated_settings.csv")
    )
    setting_dirs = sorted(
        (p for p in data_dir.glob("setting_*") if p.is_dir()),
        key=lambda p: numeric_suffix(p, "setting_"),
    )
    batch_dirs = sorted(
        (b for s in setting_dirs for b in s.glob("batch_*") if b.is_dir()),
        key=lambda p: (numeric_suffix(p.parent, "setting_"),
                       numeric_suffix(p, "batch_")),
    )
    trees_by_batch = {}
    generated_trees = []
    incomplete_tree_dirs = 0
    for batch_dir in batch_dirs:
        key = (batch_dir.parent.name, batch_dir.name)
        tree_dirs = sorted(
            (p for p in batch_dir.glob("tree_*") if p.is_dir()
             and numeric_suffix(p, "tree_") >= 0),
            key=lambda p: numeric_suffix(p, "tree_"),
        )
        complete = [p for p in tree_dirs
                    if (p / "nodes.csv").is_file() and (p / "edges.csv").is_file()]
        incomplete_tree_dirs += len(tree_dirs) - len(complete)
        trees_by_batch[key] = complete
        generated_trees.extend(complete)

    fits, correlations = [], []
    invalid = defaultdict(int)
    progressive_warnings = []

    if topology == "star":
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            comparisons = load_completed_progressive_comparisons(data_dir, 3)
        progressive_warnings = [str(w.message) for w in caught]
        for c in comparisons:
            key = (c.setting, c.batch)
            n_trees = sum(numeric_suffix(t, "tree_") < FIRST_EIGHT
                          for t in trees_by_batch.get(key, []))
            p = [c.truth[f"p_A_to_{s}"] for s in TRANSITIONS]
            estimate = [c.inferred[f"p_A_to_{s}"] for s in TRANSITIONS]
            add_fit(fits, correlations, topology, "CONCERT", c.setting,
                    c.batch, n_trees, true_values=p, estimates=estimate)

        for path in sorted(data_dir.glob(
                "setting_*/batch_*/diversitree_musse/joint_musse_fit.txt")):
            setting, batch = path.parents[2].name, path.parents[1].name
            if setting not in truth_by_setting:
                continue
            try:
                rates = read_star_diversitree(path)
            except (OSError, ValueError):
                invalid["diversitree"] += 1
                continue
            key = (setting, batch)
            p = [truth_by_setting[setting][s] for s in TRANSITIONS]
            estimate = [rates[s] for s in TRANSITIONS]
            add_fit(fits, correlations, topology, "diversitree", setting,
                    batch, len(trees_by_batch.get(key, [])),
                    true_values=p, estimates=estimate)

        for tree_dir in generated_trees:
            summary = tree_dir / "beam_snakemake" / "parameter_summary.tsv"
            if not summary.is_file():
                continue
            setting, batch, tree = (tree_dir.parents[1].name,
                                    tree_dir.parent.name, tree_dir.name)
            try:
                rates = read_star_beam_rates(summary)
            except (OSError, ValueError):
                invalid["BEAM"] += 1
                continue
            p = [truth_by_setting[setting][s] for s in TRANSITIONS]
            estimate = [rates[s] for s in TRANSITIONS]
            add_fit(fits, correlations, topology, "BEAM", setting, batch,
                    1, tree=tree, true_values=p, estimates=estimate)
    else:
        for path in sorted(data_dir.glob(
                "setting_*/batch_*/inference_joint_coordinate_first_8_trees.out")):
            setting, batch = path.parents[1].name, path.parent.name
            if setting not in truth_by_setting:
                continue
            try:
                estimates = read_linear_concert(path)
            except (OSError, ValueError):
                invalid["CONCERT"] += 1
                continue
            p = [truth_by_setting[setting][pk] for _, _, pk, _ in LINEAR_TRANSITIONS]
            estimate = [estimates[pk] for _, _, pk, _ in LINEAR_TRANSITIONS]
            key = (setting, batch)
            n_trees = sum(numeric_suffix(t, "tree_") < FIRST_EIGHT
                          for t in trees_by_batch.get(key, []))
            add_fit(fits, correlations, topology, "CONCERT", setting,
                    batch, n_trees, true_values=p, estimates=estimate)

        for path in sorted(data_dir.glob(
                "setting_*/batch_*/diversitree_musse_first8/joint_musse_fit.txt")):
            setting, batch = path.parents[2].name, path.parents[1].name
            if setting not in truth_by_setting:
                continue
            try:
                estimates = read_linear_diversitree(path)
            except (OSError, ValueError):
                invalid["diversitree"] += 1
                continue
            p = [truth_by_setting[setting][pk] for _, _, pk, _ in LINEAR_TRANSITIONS]
            estimate = [estimates[f"transition_rate_{edge}"]
                        for _, edge, _, _ in LINEAR_TRANSITIONS]
            key = (setting, batch)
            n_trees = sum(numeric_suffix(t, "tree_") < FIRST_EIGHT
                          for t in trees_by_batch.get(key, []))
            add_fit(fits, correlations, topology, "diversitree", setting,
                    batch, n_trees, true_values=p, estimates=estimate)

        for tree_dir in generated_trees:
            summary = tree_dir / "beam_snakemake" / "parameter_summary.tsv"
            if not summary.is_file():
                continue
            setting, batch, tree = (tree_dir.parents[1].name,
                                    tree_dir.parent.name, tree_dir.name)
            try:
                rates = read_linear_beam_rates(summary, LINEAR_BEAM_RATE_COLUMNS)
            except (OSError, ValueError):
                invalid["BEAM"] += 1
                continue
            p = [truth_by_setting[setting][pk] for _, _, pk, _ in LINEAR_TRANSITIONS]
            estimate = [rates[target] for target, _, _, _ in LINEAR_TRANSITIONS]
            add_fit(fits, correlations, topology, "BEAM", setting, batch,
                    1, tree=tree, true_values=p, estimates=estimate)

    mach2_no_transition_trees = 0
    for tree_dir in generated_trees:
        output_dir = tree_dir / "mach2_full" / "mach2_output"
        if not output_dir.is_dir():
            continue
        for path in sorted(output_dir.glob("*-T-0.refined.tree")):
            try:
                result = _read_mach2_tree_transition_time_error(path)
            except (OSError, ValueError):
                invalid["MACH2"] += 1
                continue
            no_transitions = result is None
            mach2_no_transition_trees += int(no_transitions)
            add_fit(fits, correlations, topology, "MACH2",
                    tree_dir.parents[1].name, tree_dir.parent.name, 1,
                    tree=tree_dir.name, no_transitions=no_transitions)

    gt_settings = settings_csv["setting"].nunique()
    expected_batches = gt_settings * BATCHES_PER_SETTING
    expected_trees = expected_batches * spec["trees_per_batch"]
    n_generated = len(generated_trees)
    nonempty_batches = sum(bool(paths) for paths in trees_by_batch.values())
    eligible_trees = [t for t in generated_trees
                      if topology == "star" or numeric_suffix(t, "tree_") < FIRST_EIGHT]
    coverage = {
        "dataset": dataset,
        "settings_in_truth": gt_settings,
        "setting_dirs": len(setting_dirs),
        "expected_batches": expected_batches,
        "batch_dirs": len(batch_dirs),
        "batches_with_trees": nonempty_batches,
        "expected_trees": expected_trees,
        "generated_trees": n_generated,
        "tree_generation_percent": 100 * n_generated / expected_trees,
        "incomplete_tree_dirs": incomplete_tree_dirs,
    }

    method_rows = []
    for method in ("CONCERT", "diversitree", "BEAM", "MACH2"):
        outputs = [fit for fit in fits if fit["method"] == method]
        is_batch_fit = method in ("CONCERT", "diversitree")
        target_units = expected_batches if is_batch_fit else len(eligible_trees)
        trees_used = sum(fit["trees_used"] for fit in outputs)
        method_rows.append({
            "dataset": dataset,
            "method": method,
            "fit_unit": "batch fits" if is_batch_fit else "tree runs",
            "completed_fits": len(outputs),
            "target_units": target_units,
            "completed_batches": len({(fit["setting"], fit["batch"])
                                       for fit in outputs}),
            "trees_in_completed_fits": trees_used,
            "fit_completion_percent": 100 * len(outputs) / target_units
            if target_units else np.nan,
            "generated_tree_coverage_percent": 100 * trees_used / n_generated
            if n_generated else np.nan,
            "MACH2_trees_without_transitions": (
                sum(fit["no_predicted_transitions"] for fit in outputs)
                if method == "MACH2" else np.nan
            ),
            "invalid_or_unparseable_outputs": invalid[method],
        })
    return {
        "coverage": coverage, "methods": method_rows, "fits": fits,
        "correlations": correlations, "warnings": progressive_warnings,
    }


snapshot = {name: scan_simulation(name) for name in TOPOLOGY_SPECS}
coverage_df = pd.DataFrame([item["coverage"] for item in snapshot.values()])
method_progress_df = pd.DataFrame(
    [row for item in snapshot.values() for row in item["methods"]]
)
correlation_runs_df = pd.DataFrame(
    [row for item in snapshot.values() for row in item["correlations"]]
)
print("Generated simulation inputs")
display(coverage_df.round(2))
print("Valid completed inference outputs; batch methods report covered input trees")
display(method_progress_df.round(2))

Generated simulation inputs


,dataset,settings_in_truth,setting_dirs,expected_batches,batch_dirs,batches_with_trees,expected_trees,generated_trees,tree_generation_percent,incomplete_tree_dirs
0,star_3,100,100,1000,1000,1000,128000,24553,19.18,25
1,linear_A_to_D,100,100,1000,1000,1000,10000,7995,79.95,7


Valid completed inference outputs; batch methods report covered input trees


,dataset,method,fit_unit,completed_fits,target_units,completed_batches,trees_in_completed_fits,fit_completion_percent,generated_tree_coverage_percent,MACH2_trees_without_transitions,invalid_or_unparseable_outputs
0,star_3,CONCERT,batch fits,145,1000,145,1160,14.50,4.72,NaN,0
1,star_3,diversitree,batch fits,70,1000,70,1722,7.00,7.01,NaN,0
2,star_3,BEAM,tree runs,530,24553,179,530,2.16,2.16,NaN,0
3,star_3,MACH2,tree runs,262,24553,117,262,1.07,1.07,90.0,0
4,linear_A_to_D,CONCERT,batch fits,234,1000,234,1872,23.40,23.41,NaN,0
5,linear_A_to_D,diversitree,batch fits,266,1000,266,2128,26.60,26.62,NaN,1
6,linear_A_to_D,BEAM,tree runs,447,7993,58,447,5.59,5.59,NaN,0
7,linear_A_to_D,MACH2,tree runs,197,7993,66,197,2.46,2.46,49.0,0


In [2]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(coverage_df["dataset"], coverage_df["tree_generation_percent"],
            color=["#4E9FCF", "#4FBB9B"])
axes[0].set_title("Generated trees / configured target")
axes[0].set_ylabel("Complete tree inputs (%)")
axes[0].set_ylim(0, 100)
axes[0].tick_params(axis="x", rotation=15)
for i, row in coverage_df.reset_index(drop=True).iterrows():
    axes[0].text(i, row["tree_generation_percent"] + 2,
                 f"{row['generated_trees']:,} / {row['expected_trees']:,}",
                 ha="center", fontsize=8)
plot_data = method_progress_df.pivot(
    index="method", columns="dataset", values="fit_completion_percent"
).reindex(["CONCERT", "diversitree", "BEAM", "MACH2"])
plot_data.plot(kind="bar", ax=axes[1], color=["#4E9FCF", "#4FBB9B"])
axes[1].set_title("Completed inference units / target")
axes[1].set_ylabel("Completed units (%)")
axes[1].set_ylim(0, max(5, axes[1].get_ylim()[1]))
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Dataset", fontsize=8)
fig.subplots_adjust(bottom=0.2, wspace=0.32)
plt.show()

## All-completed-output error summaries (exploratory)

These summaries use all currently valid outputs and are not the common 20+20 comparison used for the final figure or manuscript paragraph. The common-batch results are reported in the final section of this notebook.

In [3]:
with warnings.catch_warnings(record=True) as metric_warnings:
    warnings.simplefilter("always")
    rate_errors = {
        "star": load_matched_absolute_errors(DATA_ROOT / "star_3", num_transition_times=3),
        "linear": load_matched_linear_absolute_errors(DATA_ROOT / "linear_A_to_D"),
    }
    concert_time_errors = {
        "star": load_star_transition_time_errors(DATA_ROOT / "star_3", 3),
        "linear": load_concert_transition_time_errors(DATA_ROOT / "linear_A_to_D"),
    }
    mach2_time_errors = {
        topology: load_mach2_transition_time_errors(DATA_ROOT / dataset)
        for topology, dataset in (("star", "star_3"), ("linear", "linear_A_to_D"))
    }

rate_rows = []
for topology, errors in rate_errors.items():
    for metric in ("net_growth", "scaled_birth"):
        for method, source_method in (("CONCERT", "CONCERT"),
                                      ("diversitree", "MuSSE"),
                                      ("BEAM", "BEAM")):
            values = [e.value for e in errors
                      if e.metric == metric and e.method == source_method]
            rate_rows.append({
                "topology": topology, "metric": metric, "method": method,
                "n_error_observations": len(values),
                "median_absolute_error": float(np.median(values)) if values else np.nan,
            })
rate_error_summary = pd.DataFrame(rate_rows)

time_rows = []
for topology in ("linear", "star"):
    for method, values in (
        ("CONCERT", [e.value for e in concert_time_errors[topology]]),
        ("MACH2", [e.value for e in mach2_time_errors[topology]]),
    ):
        time_rows.append({
            "topology": topology, "method": method,
            "n_error_observations": len(values),
            "median_MAE": float(np.median(values)) if values else np.nan,
        })
time_error_summary = pd.DataFrame(time_rows)
print("Median rate absolute errors")
display(rate_error_summary.round(4))
print("Median transition-time MAEs")
display(time_error_summary.round(4))
print(f"Loader warnings recorded: {len(metric_warnings)} (incomplete fits and unscorable runs).")

Median rate absolute errors


,topology,metric,method,n_error_observations,median_absolute_error
0,star,net_growth,CONCERT,345,0.3873
1,star,net_growth,diversitree,345,0.8504
2,star,net_growth,BEAM,80,1.2848
3,star,scaled_birth,CONCERT,345,0.2964
4,star,scaled_birth,diversitree,345,0.5497
5,star,scaled_birth,BEAM,80,1.5038
6,linear,net_growth,CONCERT,465,0.6375
7,linear,net_growth,diversitree,465,1.2824
8,linear,net_growth,BEAM,415,2.1915
9,linear,scaled_birth,CONCERT,465,0.4375


Median transition-time MAEs


,topology,method,n_error_observations,median_MAE
0,linear,CONCERT,702,0.0070
1,linear,MACH2,55,0.0441
2,star,CONCERT,435,0.0050
3,star,MACH2,89,0.0462


Loader warnings recorded: 3 (incomplete fits and unscorable runs).


## Exploratory Spearman and order-error diagnostics

These all-output rank summaries are retained as diagnostics; the final 20+20 manuscript comparison uses Pearson correlations on common batches, as described in the final section.

In [4]:
from src.plot_star_diversitree_errors import load_transition_rank_correlations

rank_rows = []
for dataset, topology in (("linear_A_to_D", "linear"), ("star_3", "star")):
    for result in load_transition_rank_correlations(
        DATA_ROOT / dataset,
        topology,
        num_transition_times=3,
    ):
        rank_rows.append({
            "topology": topology,
            "method": result.method,
            "setting": result.setting,
            "batch": result.batch,
            "spearman_rho": result.value,
            "order_error_rate": result.order_error_rate,
        })
correlation_runs_df = pd.DataFrame(rank_rows)
defined_rank_runs_df = correlation_runs_df.dropna(subset=["spearman_rho"])
correlation_summary = (
    defined_rank_runs_df.groupby(["topology", "method"], as_index=False)
    .agg(
        median_spearman_rho=("spearman_rho", "median"),
        defined_batches=("spearman_rho", "size"),
    )
)
order_error_summary = (
    correlation_runs_df.groupby(["topology", "method"], as_index=False)
    .agg(
        median_wrong_order_rate=("order_error_rate", "median"),
        batches_scored=("order_error_rate", "size"),
    )
)
print("Median batch-level Spearman rho (each value ranks three transitions)")
display(correlation_summary.round(4))
print("Median fraction of pairwise transition orders wrong")
display(order_error_summary.round(4))

/home/saminrk/workspace/phylodynamic-CARTA/src/plot_star_diversitree_errors.py:908: UserWarning: Skipping incomplete diversitree output /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/linear_A_to_D/setting_4/batch_4/diversitree_musse_first8/joint_musse_fit.txt: MuSSE fit did not converge in /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/linear_A_to_D/setting_4/batch_4/diversitree_musse_first8/joint_musse_fit.txt
  warnings.warn(f"Skipping incomplete diversitree output {path}: {error}")
/localscratch/saminrk/ipykernel_2571151/2780517321.py:5: UserWarning: Spearman correlation is undefined for 39 diversitree batches; they are retained in the order-error-rate plot
  for result in load_transition_rank_correlations(


Median batch-level Spearman rho (each value ranks three transitions)


/localscratch/saminrk/ipykernel_2571151/2780517321.py:5: UserWarning: Spearman correlation is undefined for 9 diversitree batches; they are retained in the order-error-rate plot
  for result in load_transition_rank_correlations(


,topology,method,median_spearman_rho,defined_batches
0,linear,BEAM,0.5,58
1,linear,CONCERT,1.0,234
2,linear,diversitree,0.5,227
3,star,BEAM,0.5,179
4,star,CONCERT,1.0,145
5,star,diversitree,1.0,61


Median fraction of pairwise transition orders wrong


,topology,method,median_wrong_order_rate,batches_scored
0,linear,BEAM,0.3333,58
1,linear,CONCERT,0.0000,234
2,linear,diversitree,0.3333,266
3,star,BEAM,0.3333,179
4,star,CONCERT,0.0000,145
5,star,diversitree,0.0000,70


## Common-batch 20+20 results

The earlier all-output summaries above are exploratory. Use the common-batch list, statistics, and manuscript text in the final section for the figure and reported results.

In [5]:
from IPython.display import Markdown, display

display(Markdown("The all-output paragraph previously shown here is superseded by the common-batch 20+20 results at the end of this notebook."))

The all-output paragraph previously shown here is superseded by the common-batch 20+20 results at the end of this notebook.

## Most-completed settings (all available outputs)

This section ranks settings across the full on-disk results, not just the selected 20+20 batches used in the final figure.

In [6]:
from collections import Counter

from src.plot_star_diversitree_errors import load_topology_comparison_row


def setting_number(setting):
    return int(setting.rsplit("_", 1)[1])


for dataset, topology, label in (
    ("star_3", "star", "star_3"),
    ("linear_A_to_D", "linear", "linear A→D"),
):
    common_row = load_topology_comparison_row(DATA_ROOT / dataset, label, topology)
    common_batches = set(common_row.common_batch_keys)
    completed_fits = []
    for fit in snapshot[dataset]["fits"]:
        if fit["method"] == "BEAM":
            status_path = (
                DATA_ROOT / dataset / fit["setting"] / fit["batch"]
                / fit["tree"] / "beam_snakemake" / "run_status.tsv"
            )
            if not status_path.is_file():
                continue
            status_rows = pd.read_csv(status_path, sep="\t")
            if (
                len(status_rows) != 1
                or "status" not in status_rows
                or status_rows.iloc[0]["status"] != "completed"
            ):
                continue
        completed_fits.append(fit)

    records = []
    settings = sorted(
        {fit["setting"] for fit in completed_fits}, key=setting_number
    )
    for setting in settings:
        setting_fits = [fit for fit in completed_fits if fit["setting"] == setting]
        counts = Counter(fit["method"] for fit in setting_fits)
        mach2_fits = [fit for fit in setting_fits if fit["method"] == "MACH2"]
        records.append({
            "dataset": dataset,
            "setting": setting,
            "all_method_completed_runs": len(setting_fits),
            "common_completed_batches_all_data": sum(
                batch_setting == setting for batch_setting, _ in common_batches
            ),
            "CONCERT_batch_fits": counts["CONCERT"],
            "diversitree_batch_fits": counts["diversitree"],
            "BEAM_completed_tree_runs": counts["BEAM"],
            "MACH2_completed_tree_runs": counts["MACH2"],
            "MACH2_runs_without_transitions": sum(
                fit["no_predicted_transitions"] for fit in mach2_fits
            ),
            "MACH2_scorable_tree_runs": sum(
                not fit["no_predicted_transitions"] for fit in mach2_fits
            ),
        })

    records.sort(key=lambda record: (
        -record["all_method_completed_runs"],
        -record["common_completed_batches_all_data"],
        setting_number(record["setting"]),
    ))
    print(f"{dataset}: {len(common_batches)} all-data common batches")
    display(pd.DataFrame(records[:10]))

/home/saminrk/workspace/phylodynamic-CARTA/src/plot_star_diversitree_errors.py:1074: UserWarning: Spearman correlation is undefined for 9 diversitree batches; they are retained in the order-error-rate plot
  rank_correlations = load_transition_rank_correlations(data_dir, "star", 3)
/home/saminrk/workspace/phylodynamic-CARTA/src/plot_star_diversitree_errors.py:708: UserWarning: Skipped 90 MACH2 trees with no inferred transitions in /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/star_3
  warnings.warn(


star_3: 10 all-data common batches


,dataset,setting,all_method_completed_runs,common_completed_batches_all_data,CONCERT_batch_fits,diversitree_batch_fits,BEAM_completed_tree_runs,MACH2_completed_tree_runs,MACH2_runs_without_transitions,MACH2_scorable_tree_runs
0,star_3,setting_100,74,0,6,0,45,23,8,15
1,star_3,setting_11,36,0,4,2,16,14,11,3
2,star_3,setting_53,26,0,4,3,11,8,4,4
3,star_3,setting_72,24,0,2,1,12,9,2,7
4,star_3,setting_25,23,1,3,2,10,8,1,7
5,star_3,setting_67,21,0,3,2,9,7,1,6
6,star_3,setting_16,20,0,1,0,10,9,6,3
7,star_3,setting_92,19,1,1,1,10,7,0,7
8,star_3,setting_9,19,0,2,1,10,6,3,3
9,star_3,setting_13,19,0,4,2,8,5,0,5


/home/saminrk/workspace/phylodynamic-CARTA/src/plot_star_diversitree_errors.py:450: UserWarning: Skipping incomplete MuSSE output /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/linear_A_to_D/setting_4/batch_4/diversitree_musse_first8/joint_musse_fit.txt: MuSSE fit did not converge in /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/linear_A_to_D/setting_4/batch_4/diversitree_musse_first8/joint_musse_fit.txt
  warnings.warn(f"Skipping incomplete MuSSE output {summary_path}: {error}")
/home/saminrk/workspace/phylodynamic-CARTA/src/plot_star_diversitree_errors.py:908: UserWarning: Skipping incomplete diversitree output /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/linear_A_to_D/setting_4/batch_4/diversitree_musse_first8/joint_musse_fit.txt: MuSSE fit did not converge in /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/linear_A_to_D/setting_4/batch_4/diversitree_musse_first8/joint_musse_fit.txt
  warni

linear_A_to_D: 49 all-data common batches


,dataset,setting,all_method_completed_runs,common_completed_batches_all_data,CONCERT_batch_fits,diversitree_batch_fits,BEAM_completed_tree_runs,MACH2_completed_tree_runs,MACH2_runs_without_transitions,MACH2_scorable_tree_runs
0,linear_A_to_D,setting_2,95,5,7,6,47,35,4,31
1,linear_A_to_D,setting_3,44,3,5,5,22,12,0,12
2,linear_A_to_D,setting_15,37,1,5,3,16,13,6,7
3,linear_A_to_D,setting_18,27,1,6,6,7,8,1,7
4,linear_A_to_D,setting_86,24,1,6,4,8,6,2,4
5,linear_A_to_D,setting_94,24,1,5,5,8,6,2,4
6,linear_A_to_D,setting_26,22,1,4,2,8,8,4,4
7,linear_A_to_D,setting_43,22,1,4,4,8,6,0,6
8,linear_A_to_D,setting_31,21,1,4,3,8,6,2,4
9,linear_A_to_D,setting_62,21,1,5,6,8,2,0,2


In [7]:

import csv

import pandas as pd
from IPython.display import display, Markdown

from src.coord_diff_top_batches import (
    LINEAR_TOP_TWENTY_BATCHES,
    STAR3_TOP_TWENTY_BATCHES,
)
from src.plot_linear_transition_inference_accuracy import (
    _read_diversitree_estimates as read_linear_diversitree,
    _read_joint_estimates as read_linear_concert,
)
from src.plot_star_diversitree_errors import (
    LINEAR_BEAM_RATE_COLUMNS,
    _read_beam_rate_estimates,
    _read_beam_rates,
    _read_mach2_tree_transition_time_error,
)
from src.plot_star_inference_accuracy import load_completed_progressive_comparisons
from src.plot_star_transition_rates_vs_probabilities import (
    _read_diversitree_rates as read_star_diversitree,
)


def valid_output(path, reader):
    if not path.is_file():
        return False
    try:
        reader(path)
    except (OSError, ValueError):
        return False
    return True


def make_batch_status(batch_keys):
    return {
        key: {method: 0 for method in ("CONCERT", "diversitree", "BEAM", "MACH2")}
        for key in batch_keys
    }


selected_batch_sets = {
    "star_3": STAR3_TOP_TWENTY_BATCHES,
    "linear_A_to_D": LINEAR_TOP_TWENTY_BATCHES,
}
selected_batch_status = {
    dataset: make_batch_status(batch_keys)
    for dataset, batch_keys in selected_batch_sets.items()
}
mach2_no_transition_counts = {
    (dataset, key): 0
    for dataset, batch_keys in selected_batch_sets.items()
    for key in batch_keys
}

star_data_dir = DATA_ROOT / "star_3"
star_concert_keys = {
    (fit.setting.removeprefix("setting_"), fit.batch.removeprefix("batch_"))
    for fit in load_completed_progressive_comparisons(star_data_dir, 3)
}
for key in STAR3_TOP_TWENTY_BATCHES:
    selected_batch_status["star_3"][key]["CONCERT"] = int(key in star_concert_keys)
    setting, batch = key
    star_fit = (
        star_data_dir / f"setting_{setting}" / f"batch_{batch}"
        / "diversitree_musse_first8" / "joint_musse_fit.txt"
    )
    selected_batch_status["star_3"][key]["diversitree"] = int(
        valid_output(star_fit, read_star_diversitree)
    )

for setting, batch in LINEAR_TOP_TWENTY_BATCHES:
    key = (setting, batch)
    batch_dir = DATA_ROOT / "linear_A_to_D" / f"setting_{setting}" / f"batch_{batch}"
    selected_batch_status["linear_A_to_D"][key]["CONCERT"] = int(
        valid_output(
            batch_dir / "inference_joint_coordinate_first_8_trees.out",
            read_linear_concert,
        )
    )
    selected_batch_status["linear_A_to_D"][key]["diversitree"] = int(
        valid_output(
            batch_dir / "diversitree_musse_first8" / "joint_musse_fit.txt",
            read_linear_diversitree,
        )
    )

for dataset, batch_keys in selected_batch_sets.items():
    data_dir = DATA_ROOT / dataset
    beam_reader = (
        _read_beam_rate_estimates
        if dataset == "star_3"
        else lambda path: _read_beam_rates(path, LINEAR_BEAM_RATE_COLUMNS)
    )
    for setting, batch in batch_keys:
        key = (setting, batch)
        for tree in range(8):
            tree_dir = data_dir / f"setting_{setting}" / f"batch_{batch}" / f"tree_{tree}"
            summary_path = tree_dir / "beam_snakemake" / "parameter_summary.tsv"
            status_path = tree_dir / "beam_snakemake" / "run_status.tsv"
            if status_path.is_file():
                with status_path.open(newline="") as source:
                    status_rows = list(csv.DictReader(source, delimiter="\t"))
                if (
                    len(status_rows) == 1
                    and status_rows[0].get("status") == "completed"
                    and valid_output(summary_path, beam_reader)
                ):
                    selected_batch_status[dataset][key]["BEAM"] += 1

            mach2_outputs = sorted(
                (tree_dir / "mach2_full" / "mach2_output").glob(
                    "*-T-0.refined.tree"
                )
            )
            if len(mach2_outputs) == 1:
                try:
                    result = _read_mach2_tree_transition_time_error(mach2_outputs[0])
                except (OSError, ValueError):
                    continue
                selected_batch_status[dataset][key]["MACH2"] += 1
                mach2_no_transition_counts[(dataset, key)] += int(result is None)


detail_rows = []
summary_rows = []
for dataset, batches in selected_batch_status.items():
    completed_outputs = 0
    fully_completed_batches = 0
    for (setting, batch), counts in batches.items():
        completed = sum(counts.values())
        completed_outputs += completed
        fully_completed_batches += int(completed == 18)
        detail_rows.append({
            "dataset": dataset,
            "setting": f"setting_{setting}",
            "batch": f"batch_{batch}",
            "CONCERT (1)": counts["CONCERT"],
            "diversitree (1)": counts["diversitree"],
            "BEAM (8)": counts["BEAM"],
            "MACH2 (8)": counts["MACH2"],
            "completed outputs": f"{completed}/18",
            "completion percent": 100 * completed / 18,
            "MACH2 no-transition runs": mach2_no_transition_counts[(dataset, (setting, batch))],
        })
    expected_outputs = len(batches) * 18
    summary_rows.append({
        "dataset": dataset,
        "selected batches": len(batches),
        "fully completed batches": fully_completed_batches,
        "completed method outputs": completed_outputs,
        "expected method outputs": expected_outputs,
        "completion percent": 100 * completed_outputs / expected_outputs,
        "outputs remaining": expected_outputs - completed_outputs,
    })

display(Markdown("## Selected-batch completion (first eight trees)"))
display(pd.DataFrame(summary_rows).round(2))
display(pd.DataFrame(detail_rows).round({"completion percent": 1}))


## Selected-batch completion (first eight trees)

,dataset,selected batches,fully completed batches,completed method outputs,expected method outputs,completion percent,outputs remaining
0,star_3,20,1,296,360,82.22,64
1,linear_A_to_D,20,0,280,360,77.78,80


,dataset,setting,batch,CONCERT (1),diversitree (1),BEAM (8),MACH2 (8),completed outputs,completion percent,MACH2 no-transition runs
0,star_3,setting_53,batch_0,1,1,8,6,16/18,88.9,2
1,star_3,setting_56,batch_3,1,1,8,4,14/18,77.8,0
2,star_3,setting_57,batch_9,1,1,8,6,16/18,88.9,3
3,star_3,setting_72,batch_9,1,1,8,6,16/18,88.9,1
4,star_3,setting_86,batch_1,1,1,7,6,15/18,83.3,3
5,star_3,setting_92,batch_7,1,1,8,6,16/18,88.9,0
6,star_3,setting_6,batch_6,0,1,8,5,14/18,77.8,5
7,star_3,setting_9,batch_0,1,1,8,5,15/18,83.3,2
8,star_3,setting_10,batch_4,1,1,8,7,17/18,94.4,0
9,star_3,setting_11,batch_5,1,1,8,7,17/18,94.4,4


In [8]:
from IPython.display import Markdown, display

selected_batch_count = sum(len(batch_keys) for batch_keys in selected_batch_sets.values())
tree_runs_per_batch = 8
expected_runs_by_method = {
    "CONCERT": selected_batch_count,
    "diversitree": selected_batch_count,
    "BEAM": selected_batch_count * tree_runs_per_batch,
    "MACH2": selected_batch_count * tree_runs_per_batch,
}
completed_runs_by_method = {
    method: sum(
        counts[method]
        for dataset_status in selected_batch_status.values()
        for counts in dataset_status.values()
    )
    for method in expected_runs_by_method
}

remaining_rows = []
for method, expected_runs in expected_runs_by_method.items():
    completed_runs = completed_runs_by_method[method]
    if completed_runs > expected_runs:
        raise ValueError(
            f"{method} has {completed_runs} completed runs, exceeding "
            f"the expected {expected_runs}"
        )
    runs_left = expected_runs - completed_runs
    remaining_rows.append({
        "method": method,
        "completed runs": completed_runs,
        "expected runs": expected_runs,
        "runs left": runs_left,
        "runs left (%)": 100 * runs_left / expected_runs,
    })

remaining_runs_df = pd.DataFrame(remaining_rows)
display(Markdown(
    "## Runs remaining for the selected 20+20 batches\n\n"
    "CONCERT and diversitree each have one fit per batch; BEAM and MACH2 each "
    "have up to eight tree-level runs per batch. A valid completed MACH2 run "
    "counts as complete even if it inferred no transitions."
))
display(remaining_runs_df.round({"runs left (%)": 1}))

## Runs remaining for the selected 20+20 batches

CONCERT and diversitree each have one fit per batch; BEAM and MACH2 each have up to eight tree-level runs per batch. A valid completed MACH2 run counts as complete even if it inferred no transitions.

,method,completed runs,expected runs,runs left,runs left (%)
0,CONCERT,37,40,3,7.5
1,diversitree,40,40,0,0.0
2,BEAM,312,320,8,2.5
3,MACH2,187,320,133,41.6


In [9]:
from statistics import median

from IPython.display import Markdown, display

from src.coord_diff_top_batches import (
    LINEAR_TOP_TWENTY_BATCHES,
    STAR3_TOP_TWENTY_BATCHES,
)
from src.plot_star_diversitree_errors import (
    _topology_error_values,
    load_selected_topology_comparison_row,
)

common_plot_rows = {}
common_batch_details = []
common_statistic_rows = []
for dataset, topology, label, selection in (
    ("linear_A_to_D", "linear", "linear A→D", LINEAR_TOP_TWENTY_BATCHES),
    ("star_3", "star", "star_3", STAR3_TOP_TWENTY_BATCHES),
):
    row = load_selected_topology_comparison_row(
        DATA_ROOT / dataset, label, topology, selection
    )
    common_plot_rows[topology] = row
    expected_common = {
        (f"setting_{setting}", f"batch_{batch}")
        for (setting, batch), counts in selected_batch_status[dataset].items()
        if counts["CONCERT"] == 1
        and counts["diversitree"] == 1
        and counts["BEAM"] >= 1
        and counts["MACH2"] >= 1
    }
    actual_common = set(row.common_batch_keys)
    if actual_common != expected_common:
        raise AssertionError(
            f"{dataset} figure batches do not match the recorded completion "
            f"criteria; only in status={sorted(expected_common - actual_common)}, "
            f"only in figure={sorted(actual_common - expected_common)}"
        )

    for setting, batch in sorted(
        actual_common,
        key=lambda key: (int(key[0].split("_")[-1]), int(key[1].split("_")[-1])),
    ):
        status = selected_batch_status[dataset][
            (setting.removeprefix("setting_"), batch.removeprefix("batch_"))
        ]
        common_batch_details.append({
            "topology": topology,
            "setting": setting,
            "batch": batch,
            "CONCERT completed": bool(status["CONCERT"]),
            "diversitree completed": bool(status["diversitree"]),
            "BEAM completed trees (of 8)": status["BEAM"],
            "MACH2 completed trees (of 8)": status["MACH2"],
            "MACH2 trees without transitions": mach2_no_transition_counts[
                (dataset, (setting.removeprefix("setting_"), batch.removeprefix("batch_")))
            ],
        })

    for metric, methods in (
        ("net_growth", ("CONCERT", "diversitree", "BEAM")),
        ("scaled_birth", ("CONCERT", "diversitree", "BEAM")),
        ("transition_correlation", ("CONCERT", "diversitree", "BEAM")),
        ("transition_time", ("CONCERT", "MACH2")),
    ):
        for method in methods:
            values = _topology_error_values(row, metric, method)
            if values:
                common_statistic_rows.append({
                    "topology": topology,
                    "metric": metric,
                    "method": method,
                    "n estimates": len(values),
                    "median": median(values),
                })

common_batches_df = pd.DataFrame(common_batch_details)
common_statistics_df = pd.DataFrame(common_statistic_rows)
display(Markdown(
    "## Exact batches and statistics used in the 20+20 figure and paragraph\n\n"
    "A batch is included only when CONCERT and diversitree each have a valid fit, "
    "and at least one valid BEAM and MACH2 tree run is present. A valid MACH2 run "
    "with no inferred transition still qualifies as completed, but contributes no "
    "transition-time error. The figure is loaded through this same common-batch filter."
))
print("Common-batch counts:", {
    topology: len(row.common_batch_keys)
    for topology, row in common_plot_rows.items()
})
display(common_batches_df)
display(common_statistics_df.round({"median": 3}))

def common_median(topology, metric, method):
    values = common_statistics_df.loc[
        (common_statistics_df["topology"] == topology)
        & (common_statistics_df["metric"] == metric)
        & (common_statistics_df["method"] == method),
        "median",
    ]
    if len(values) != 1:
        raise ValueError(
            f"Expected one common-batch median for {topology}/{metric}/{method}; "
            f"found {len(values)}."
        )
    return float(values.iloc[0])


def common_comparison(topology, metric, methods):
    values = [common_median(topology, metric, method) for method in methods]
    if len(values) == 3:
        return f"{values[0]:.3f} vs. {values[1]:.3f} and {values[2]:.3f}"
    if len(values) == 2:
        return f"{values[0]:.3f} vs. {values[1]:.3f}"
    raise ValueError(f"Expected two or three methods, found {len(values)}.")


method_order = ("CONCERT", "diversitree", "BEAM")
rate_text = (
    f"\\ourmethod outperformed \\texttt{{diversitree}} and BEAM in inferring "
    f"net growth rates (median absolute error for linear: "
    f"{common_comparison('linear', 'net_growth', method_order)}; star: "
    f"{common_comparison('star', 'net_growth', method_order)}) and scaled "
    f"birth rates (linear: "
    f"{common_comparison('linear', 'scaled_birth', method_order)}; star: "
    f"{common_comparison('star', 'scaled_birth', method_order)})."
)
correlation_text = (
    f"The median correlations were higher for \\ourmethod (linear: "
    f"{common_comparison('linear', 'transition_correlation', method_order)}; "
    f"star: {common_comparison('star', 'transition_correlation', method_order)}), "
    "indicating better recovery of relative transition tendencies."
)
time_text = (
    f"Finally, \\ourmethod inferred transition times more accurately than MACH2 "
    f"(Fig.~\\ref{{fig:simulation}}; median absolute transition-time error: "
    f"linear, {common_comparison('linear', 'transition_time', ('CONCERT', 'MACH2'))}; "
    f"star, {common_comparison('star', 'transition_time', ('CONCERT', 'MACH2'))})."
)
mach2_no_transition_fits = int(
    common_batches_df["MACH2 trees without transitions"].sum()
)
mach2_note = (
    " Some completed MACH2 runs inferred no transitions and therefore do not "
    "contribute a transition-time error."
    if mach2_no_transition_fits
    else ""
)
manuscript_text = (
    "## Manuscript text for common batches\\n\\n"
    + rate_text
    + "\\n\\nSince both \\texttt{diversitree} and BEAM estimate continuous-time "
    "transition rates rather than transition probabilities, we evaluated "
    "recovery of transition dynamics using correlation between the ground-truth "
    "probabilities and the inferred rates or probabilities from each method. "
    r"For \ourmethod, inferred probabilities were used directly for linear graphs "
    r"and adjusted as \(p_{0,i}\rho_i/\rho_0\) for star graphs using the "
    r"inferred sampling probabilities to account for non-identifiability "
    r"(Theorem~\ref{thm:star-identifiability}). For \texttt{diversitree} "
    "and BEAM, we used their inferred transition rates. "
    + correlation_text
    + "\\n\\n"
    + time_text
    + mach2_note
    + " These results support the value of explicitly modeling coordinated "
    "transitions when inferring cellular dynamics from data generated under CSTM."
)
display(Markdown(manuscript_text))

/home/saminrk/workspace/phylodynamic-CARTA/src/plot_star_diversitree_errors.py:708: UserWarning: Skipped 49 MACH2 trees with no inferred transitions in /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/linear_A_to_D
  warnings.warn(
/home/saminrk/workspace/phylodynamic-CARTA/src/plot_star_diversitree_errors.py:708: UserWarning: Skipped 74 MACH2 trees with no inferred transitions in /home/saminrk/workspace/phylodynamic-CARTA/data/simulation/coord_diff/star_3
  warnings.warn(


## Exact batches and statistics used in the 20+20 figure and paragraph

A batch is included only when CONCERT and diversitree each have a valid fit, and at least one valid BEAM and MACH2 tree run is present. A valid MACH2 run with no inferred transition still qualifies as completed, but contributes no transition-time error. The figure is loaded through this same common-batch filter.

Common-batch counts: {'linear': 19, 'star': 17}


,topology,setting,batch,CONCERT completed,diversitree completed,BEAM completed trees (of 8),MACH2 completed trees (of 8),MACH2 trees without transitions
0,linear,setting_2,batch_4,True,True,8,7,0
1,linear,setting_3,batch_1,True,True,8,6,0
2,linear,setting_4,batch_9,True,True,8,2,2
3,linear,setting_6,batch_2,True,True,8,1,0
4,linear,setting_9,batch_1,True,True,8,2,1
5,linear,setting_10,batch_5,True,True,8,6,0
6,linear,setting_13,batch_7,True,True,8,1,0
7,linear,setting_15,batch_0,True,True,8,6,4
8,linear,setting_17,batch_5,True,True,8,2,1
9,linear,setting_18,batch_1,True,True,7,7,1


,topology,metric,method,n estimates,median
0,linear,net_growth,CONCERT,76,0.480
1,linear,net_growth,diversitree,76,1.615
2,linear,net_growth,BEAM,151,2.123
3,linear,scaled_birth,CONCERT,76,0.431
4,linear,scaled_birth,diversitree,76,0.662
5,linear,scaled_birth,BEAM,151,1.415
6,linear,transition_correlation,CONCERT,19,0.972
7,linear,transition_correlation,diversitree,19,0.553
8,linear,transition_correlation,BEAM,19,0.465
9,linear,transition_time,CONCERT,57,0.012


## Manuscript text for common batches\n\n\ourmethod outperformed \texttt{diversitree} and BEAM in inferring net growth rates (median absolute error for linear: 0.480 vs. 1.615 and 2.123; star: 0.370 vs. 0.656 and 1.892) and scaled birth rates (linear: 0.431 vs. 0.662 and 1.415; star: 0.308 vs. 0.461 and 1.376).\n\nSince both \texttt{diversitree} and BEAM estimate continuous-time transition rates rather than transition probabilities, we evaluated recovery of transition dynamics using correlation between the ground-truth probabilities and the inferred rates or probabilities from each method. For \ourmethod, inferred probabilities were used directly for linear graphs and adjusted as \(p_{0,i}\rho_i/\rho_0\) for star graphs using the inferred sampling probabilities to account for non-identifiability (Theorem~\ref{thm:star-identifiability}). For \texttt{diversitree} and BEAM, we used their inferred transition rates. The median correlations were higher for \ourmethod (linear: 0.972 vs. 0.553 and 0.465; star: 0.956 vs. 0.850 and 0.905), indicating better recovery of relative transition tendencies.\n\nFinally, \ourmethod inferred transition times more accurately than MACH2 (Fig.~\ref{fig:simulation}; median absolute transition-time error: linear, 0.012 vs. 0.055; star, 0.005 vs. 0.060). Some completed MACH2 runs inferred no transitions and therefore do not contribute a transition-time error. These results support the value of explicitly modeling coordinated transitions when inferring cellular dynamics from data generated under CSTM.